In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os
from datetime import datetime

from tensorflow import keras
from sklearn.preprocessing import StandardScaler

import yfinance as yf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score





C:\Users\JH\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
data = yf.download("AAPL", start="2010-01-01")
#data.head()

[*********************100%***********************]  1 of 1 completed


In [4]:
#LSTM MOdel Prepration
#Stock_Close = data.filter(["close"])
Stock_Close = data['Close']
Stock_Close.head()

dataset = Stock_Close.values # convert to numpy array
training_data_len = int(np.ceil(len(dataset) * 0.9))
# Preprocessing Stages
scaler = StandardScaler()
scaled_data = scaler.fit_transform(dataset)
training_data = scaled_data[:training_data_len] #90% of all our data

X_train, y_train = [], []

# create a sliding window for our stock (30 days)
for i in range(30, len(training_data)):
    X_train.append(training_data[i-30:i, 0])
    y_train.append(training_data[i,0])

# this is for TensorFlow because it handles arrays
X_train, y_train = np.array(X_train), np.array(y_train)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [5]:
#prepare test data
test_data = scaled_data[training_data_len-30:]
X_test, y_test = [] , dataset[training_data_len:]

for i in range(30, len(test_data)):
    X_test.append(test_data[i-30:i, 0])

X_test =  np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

#plotting data
train = data[:training_data_len]
test = data[training_data_len:]

test = test.copy()
#test['predictions'] = prerdictions

y_test_real = test.iloc[:, 0]

In [6]:
# Folder to save models
os.makedirs("saved_models", exist_ok=True)

# Dictionary to store models and metrics
model_results = {}




configs = [
    {"epochs": 20, "batch_size": 32},
    {"epochs": 30, "batch_size": 32},
    {"epochs": 40, "batch_size": 64},
]

for i, cfg in enumerate(configs, 1):
#for cfg in configs:
    print(f"Training with epochs={cfg['epochs']} batch_size={cfg['batch_size']}")
    #model.fit(X_train_scaled, y_train, epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=0)
    
    # Build the model
    model = keras.models.Sequential()
    
    # اول یک لایه Input اضافه کن
    # model.add(Input(shape=(X_train.shape[1], 1)))
    #First Layer
    #model.add(keras.layers.LSTM(64, return_sequences=True, input_shape = (X_train.shape[1],1)))
    model.add(LSTM(64, return_sequences=True, input_shape = (X_train.shape[1],1)))
    
    #Second Layer
    model.add(LSTM(64, return_sequences=False))
    
    # 3rd Layer (Dense)
    model.add(keras.layers.Dense(128, activation="relu"))
    
    #4th layer (Dropout)
    model.add(keras.layers.Dropout(0.5))
    
    # Final output layer
    model.add(keras.layers.Dense(1))
    
    #model.summary()
    model.compile(optimizer="adam", loss="mae", metrics=[keras.metrics.RootMeanSquaredError()])


    
    traning = model.fit(X_train, y_train, epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=0)
    # make a prediction
    prerdictions = model.predict(X_test)
    y_pred_real = scaler.inverse_transform(prerdictions)
    
    #y_pred_real  = prerdictions

    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    mae  = mean_absolute_error(y_test_real, y_pred_real)
    r2   = r2_score(y_test_real, y_pred_real)
    rmse_percent = (rmse / y_test_real.mean()) * 100
    # Save model to disk
    model_filename = f"saved_models/lstm_model_{i}.keras"
    model.save(model_filename)
    
    # Store everything in dictionary
    model_results[f"model_{i}"] = {
        "model": model,
        "config": cfg,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "rmse_percent": rmse_percent,
        "saved_path": model_filename
    }
    
    print(f"Model {i} saved as {model_filename} → RMSE %: {rmse_percent:.2f}%\n")
    
print("✅ All models trained, saved, and metrics stored in dictionary!")







C:\Users\JH\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training with epochs=20 batch_size=32
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step
Model 1 saved as saved_models/lstm_model_1.keras → RMSE %: 6.09%

Training with epochs=30 batch_size=32


C:\Users\JH\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
Model 2 saved as saved_models/lstm_model_2.keras → RMSE %: 5.88%

Training with epochs=40 batch_size=64


C:\Users\JH\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
Model 3 saved as saved_models/lstm_model_3.keras → RMSE %: 3.70%

✅ All models trained, saved, and metrics stored in dictionary!


In [7]:
"""    print(f"RMSE: {rmse:.3f}")
    print(f"MAE : {mae:.3f}")
    print(f"R²  : {r2:.3f}")

    mean_price = y_test_real.mean()
    print("Mean test price:", mean_price)

    rmse_percent = (rmse / mean_price) * 100
    print("RMSE %:", rmse_percent)
    print("Next")
"""

'    print(f"RMSE: {rmse:.3f}")\n    print(f"MAE : {mae:.3f}")\n    print(f"R²  : {r2:.3f}")\n\n    mean_price = y_test_real.mean()\n    print("Mean test price:", mean_price)\n\n    rmse_percent = (rmse / mean_price) * 100\n    print("RMSE %:", rmse_percent)\n    print("Next")\n'

In [8]:
#model_results

In [9]:
# Automatically select the best model
best_model_key = min(model_results, key=lambda k: model_results[k]["rmse_percent"])
best_model_info = model_results[best_model_key]

print("Best model:", best_model_key)
print("Config:", best_model_info["config"])
print("RMSE %:", best_model_info["rmse_percent"])

Best model: model_3
Config: {'epochs': 40, 'batch_size': 64}
RMSE %: 3.7034022992551234


In [11]:
import mlflow
import mlflow.keras

# -----------------------------
# MLflow setup
# -----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("LSTM_Financial_Forecasting")

# -----------------------------
# Log all models to MLflow
# -----------------------------
for key, info in model_results.items():
    with mlflow.start_run(run_name=key):
        # Log hyperparameters
        mlflow.log_params(info["config"])
        
        # Log metrics
        mlflow.log_metric("rmse", info["rmse"])
        mlflow.log_metric("mae", info["mae"])
        mlflow.log_metric("r2", info["r2"])
        mlflow.log_metric("rmse_percent", info["rmse_percent"])
        
        # Log the model artifact
        mlflow.keras.log_model(info["model"], artifact_path="model")
        print(f"{key} logged to MLflow with RMSE %: {info['rmse_percent']:.2f}%")


2025/12/28 17:49:46 INFO mlflow.tracking.fluent: Experiment with name 'LSTM_Financial_Forecasting' does not exist. Creating a new experiment.
2025/12/28 17:49:46 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:

model_1 logged to MLflow with RMSE %: 6.09%
🏃 View run model_1 at: http://127.0.0.1:5000/#/experiments/2/runs/e94d186aba484757b14600c138a3b6dd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2025/12/28 17:50:03 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2025/12/28 17:50:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


model_2 logged to MLflow with RMSE %: 5.88%
🏃 View run model_2 at: http://127.0.0.1:5000/#/experiments/2/runs/7ff06f6c3bde4bfa8e44ff2b40589b75
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2025/12/28 17:50:14 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


model_3 logged to MLflow with RMSE %: 3.70%
🏃 View run model_3 at: http://127.0.0.1:5000/#/experiments/2/runs/a872ca7f34ba4b40b617c4d71a7614a4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [12]:
# -----------------------------
# Automatically select and register best model
# -----------------------------
best_key = min(model_results, key=lambda k: model_results[k]["rmse_percent"])
best_info = model_results[best_key]
print("\nBest model:", best_key)
print("Config:", best_info["config"])
print("RMSE %:", best_info["rmse_percent"])

with mlflow.start_run(run_name=f"Best_{best_key}_Register"):
    # Log metrics and parameters again (optional)
    mlflow.log_params(best_info["config"])
    mlflow.log_metric("rmse", best_info["rmse"])
    mlflow.log_metric("mae", best_info["mae"])
    mlflow.log_metric("r2", best_info["r2"])
    mlflow.log_metric("rmse_percent", best_info["rmse_percent"])
    
    # Log the model artifact
    mlflow.keras.log_model(best_info["model"], artifact_path="model")
    
    # Register the model in MLflow Registry
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    mlflow.register_model(model_uri, "Best_LSTM_Model")
    
    print(f"✅ Best model {best_key} registered in MLflow!")

2025/12/28 17:51:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/28 17:51:32 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.



Best model: model_3
Config: {'epochs': 40, 'batch_size': 64}
RMSE %: 3.7034022992551234


Successfully registered model 'Best_LSTM_Model'.
2025/12/28 17:51:41 WARNING mlflow.tracking._model_registry.fluent: Run with id afbd5a2ee1e8480a99d984a0f6b27d62 has no artifacts at artifact path 'model', registering model based on models:/m-82ebbff17dcd43d3b60361fa9dc40025 instead
2025/12/28 17:51:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Best_LSTM_Model, version 1
Created version '1' of model 'Best_LSTM_Model'.


✅ Best model model_3 registered in MLflow!
🏃 View run Best_model_3_Register at: http://127.0.0.1:5000/#/experiments/2/runs/afbd5a2ee1e8480a99d984a0f6b27d62
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
